In [1]:
import arcpy
import numpy as np
import random
import math
import os

# 设置工作空间
arcpy.env.workspace = r"D:\ArcGIS\sunleigangPro\模拟退火"
arcpy.env.overwriteOutput = True

# 输入栅格文件
input_raster = "最终ok.tif"

# 输出点要素类
output_points = "best_points.shp"
points = []

# 点的高度
point_height = 500  # 单位：米

# 视域范围（半径）
view_distance = 100  # 单位：米

# 目标可见区域比例
target_coverage = 0.9  # 90%


# 将栅格数据加载到 NumPy 数组中
def raster_to_array(raster):
    raster_array = arcpy.RasterToNumPyArray(raster, nodata_to_value=0)
    return raster_array

# 计算栅格的有效面积
def calculate_raster_area(raster):
    cell_size = float(arcpy.GetRasterProperties_management(raster, "CELLSIZEX").getOutput(0))
    raster_array = raster_to_array(raster)
    valid_cell_count = np.count_nonzero(raster_array)
    raster_area = valid_cell_count * (cell_size ** 2)
    return raster_area

# 获取栅格的有效区域多边形
def get_raster_domain(raster):
    # 将栅格的有效区域转换为多边形
    domain_polygon = os.path.join(arcpy.env.workspace, "raster_domain.shp")
    arcpy.RasterDomain_3d(raster, domain_polygon, "POLYGON")
    return domain_polygon

# 计算初始点的数量
def calculate_initial_points(raster_area, view_distance, target_coverage):
    # 每个点的视域范围面积（圆形区域）
    view_area = math.pi * (view_distance ** 2)  # 视域范围为圆形
    # 初始点数量 = 栅格面积 * 可见比例 / 视域范围面积
    initial_points = int(raster_area/ view_area)
    return initial_points


# 计算视域覆盖率
def calculate_coverage():
    
    # 执行视域分析
    viewshed_result = arcpy.sa.Viewshed2(
        in_raster=input_raster,
        in_observer_features="temp_points.shp",
        out_agl_raster=None,
        analysis_type="FREQUENCY",
        vertical_error="0 Meters",
        out_observer_region_relationship_table=None,
        refractivity_coefficient=0.13,
        surface_offset="0 Meters",
        observer_elevation=point_height,
        observer_offset="1 Meters",
        inner_radius=None,
        inner_radius_is_3d="GROUND",
        outer_radius=view_distance,
        outer_radius_is_3d="GROUND",
        horizontal_start_angle=0,
        horizontal_end_angle=360,
        vertical_upper_angle=90,
        vertical_lower_angle=-90,
        analysis_method="ALL_SIGHTLINES",
        analysis_target_device="GPU_THEN_CPU"
    )
    
    # 将结果转换为 NumPy 数组
    viewshed_array = raster_to_array(viewshed_result)
    
    # 计算可见区域面积
    unique_values, counts = np.unique(viewshed_array, return_counts=True)
    cell_size = float(arcpy.GetRasterProperties_management(input_raster, "CELLSIZEX").getOutput(0))
    visible_area = 0
    for value, count in zip(unique_values, counts):
        if value > 0:  # 视域值大于0表示可见区域
            visible_area += count * (cell_size ** 2)
    
    # 计算覆盖率
    coverage = visible_area / raster_area
    print(f"当前覆盖率为: {coverage}")
    return coverage

# random_max算法
def random_max(num_points):
    
    # 删除旧的 temp_points.shp 文件
    temp_points = os.path.join(arcpy.env.workspace, "temp_points.shp")
    if arcpy.Exists(temp_points):
        arcpy.management.Delete(temp_points)
        
    best_coverage = 0
    
    # 定义要使用的不同bin_shape形状
    bin_shapes = ["HEXAGON", "TRANSVERSE_HEXAGON", "SQUARE", "DIAMOND", "TRIANGLE"]  # 可以根据需要修改形状HEXAGON

    # 循环执行
    for bin_shape in bin_shapes:
        # 构造输出文件路径
        # out_features = f"temp_points_{bin_shape.lower()}.shp"  # 生成不同形状的文件名

        # 创建空间采样（系统），创建新的temp_points.shp 文件
        with arcpy.EnvManager(randomGenerator="0 STANDARD_C"):
            arcpy.management.CreateSpatialSamplingLocations(
                in_study_area="raster_domain",
                out_features=r"temp_points",
                sampling_method="SYSTEMATIC",
                strata_id_field=None,
                strata_count_method="EQUAL",
                bin_shape=bin_shape,
                bin_size=num_points,
                h3_resolution=7,
                num_samples=100,
                num_samples_per_strata=100,
                population_field=None,
                geometry_type="POINT",
                min_distance="0 Meters",
                spatial_relationship="HAVE_THEIR_CENTER_IN"
            )
        current_coverage = calculate_coverage()
        if current_coverage > best_coverage:
            best_coverage = current_coverage
            # 将 temp_points.shp 复制为 best_points.shp
            arcpy.management.CopyFeatures("temp_points.shp", "best_points.shp")
    return best_coverage

# 动态调整点数以满足目标覆盖率
def optimize_coverage():
    # 初始点数
    num_points = calculate_initial_points(raster_area, view_distance, target_coverage)
    print(f"初始点数量: {num_points}")
    
    best_coverage = 0
    
    while(best_coverage < target_coverage):
        best_coverage = random_max(num_points)
        if best_coverage < target_coverage:
            print(f"当前最优覆盖率 {best_coverage * 100}% 未达到目标，尝试点数：{num_points+1}")
            num_points += 1  
        else:
            print(f"当前最优覆盖率 {best_coverage * 100}% 达到目标，点数为：{num_points}")

    return  num_points, best_coverage

def extract_and_add_coordinates(output_points):
    # 读取生成的点文件，提取点坐标
    with arcpy.da.SearchCursor(output_points, ["SHAPE@XY"]) as cursor:
        for row in cursor:
            x, y = row[0]  # 提取点的 X 和 Y 坐标
            if not math.isnan(x) and not math.isnan(y):  # 检查坐标是否有效
                points.append((x, y))  # 将坐标添加到列表中
    
    # 添加 X 和 Y 坐标字段
    arcpy.management.AddField(output_points, "X", "DOUBLE")
    arcpy.management.AddField(output_points, "Y", "DOUBLE")
    
    # 更新点数据，将坐标写入 X 和 Y 字段
    with arcpy.da.UpdateCursor(output_points, ["SHAPE@XY", "X", "Y"]) as cursor:
        for i, row in enumerate(cursor):
            if i < len(points):  # 确保不超出 points 列表范围
                x, y = points[i]
                row[1] = x  # 更新 X 字段
                row[2] = y  # 更新 Y 字段
                cursor.updateRow(row)  # 提交更新
                        
# 主程序
if __name__ == "__main__":
    # 获取栅格的有效面积
    raster_area = calculate_raster_area(input_raster)
    print(f"栅格的有效面积: {raster_area} 平方米")
    
    # 获取栅格的有效区域多边形
    domain_polygon = get_raster_domain(input_raster)
    
    # 运行动态调整点数的优化算法
    num_points, best_coverage = optimize_coverage()
    
    extract_and_add_coordinates(output_points)  
    
    # 输出结果
    print(f"最优点的数量: {num_points}")
    print(f"最优点的位置: {points}")
    print(f"可见区域覆盖率: {best_coverage * 100}%")

    # 将生成的要素类加载到地图中
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    map = aprx.listMaps()[0]  # 获取第一个地图
    map.addDataFromPath(os.path.join(arcpy.env.workspace, output_points))
    
    print("处理完成！")

栅格的有效面积: 572800.0 平方米
初始点数量: 18
当前覆盖率为: 0.690467877094972
当前覆盖率为: 0.6840083798882681
当前覆盖率为: 0.674231843575419
当前覆盖率为: 0.6480446927374302
当前覆盖率为: 0.6258729050279329
当前最优覆盖率 69.04678770949721% 未达到目标，尝试点数：19
当前覆盖率为: 0.7377793296089385
当前覆盖率为: 0.7142108938547486
当前覆盖率为: 0.6522346368715084
当前覆盖率为: 0.6751047486033519
当前覆盖率为: 0.6243016759776536
当前最优覆盖率 73.77793296089385% 未达到目标，尝试点数：20
当前覆盖率为: 0.7133379888268156
当前覆盖率为: 0.7554120111731844
当前覆盖率为: 0.6965782122905028
当前覆盖率为: 0.7138617318435754
当前覆盖率为: 0.6546787709497207
当前最优覆盖率 75.54120111731844% 未达到目标，尝试点数：21
当前覆盖率为: 0.7318435754189944
当前覆盖率为: 0.790677374301676
当前覆盖率为: 0.7506983240223464
当前覆盖率为: 0.7220670391061452
当前覆盖率为: 0.6820879888268156
当前最优覆盖率 79.0677374301676% 未达到目标，尝试点数：22
当前覆盖率为: 0.7267807262569832
当前覆盖率为: 0.7486033519553073
当前覆盖率为: 0.7655377094972067
当前覆盖率为: 0.7646648044692738
当前覆盖率为: 0.697800279329609
当前最优覆盖率 76.55377094972067% 未达到目标，尝试点数：23
当前覆盖率为: 0.7445879888268156
当前覆盖率为: 0.7506983240223464
当前覆盖率为: 0.7503491620111732
当前覆盖率为: 0.76